# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:**  Lipi Singhal
**`Roll Number`:**  U20230097
**`GitHub Branch`:** lipi_U20230097  

# Imports and Setup

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from rlcmab_sampler import sampler


# Load Datasets

In [5]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print("News Articles Dataset Shape:", news_df.shape)
print("Train Users Dataset Shape:", train_users.shape)
print("Test Users Dataset Shape:", test_users.shape)
print("\n\nNews Categories:")
print(news_df['category'].value_counts())
print("\nTrain User Categories:")
print(train_users['label'].value_counts())


News Articles Dataset Shape: (209527, 6)
Train Users Dataset Shape: (2000, 33)
Test Users Dataset Shape: (2000, 32)


News Categories:
category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS       

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [6]:
# Checking for missing values
print("Missing values in news_df:")
print(news_df.isnull().sum())
print("\nMissing values in train_users:")
print(train_users.isnull().sum())
print("\nMissing values in test_users:")
print(test_users.isnull().sum())

# Clean news articles - drop rows with missing category
news_df_clean = news_df.dropna(subset=['category', 'headline']).reset_index(drop=True)

Missing values in news_df:
link                     0
headline                 6
category                 0
short_description    19712
authors              37418
date                     0
dtype: int64

Missing values in train_users:
user_id                          0
age                            698
income                           0
clicks                           0
purchase_amount                  0
session_duration                 0
content_variety                  0
engagement_score                 0
num_transactions                 0
avg_monthly_spend                0
avg_cart_value                   0
browsing_depth                   0
revisit_rate                     0
scroll_activity                  0
time_on_site                     0
interaction_count                0
preferred_price_range            0
discount_usage_rate              0
wishlist_size                    0
product_views                    0
repeat_purchase_gap (days)       0
churn_risk_score               

In [8]:
# Category mapping - 4 categories: Entertainment, Education, Tech, Crime
category_mapping = {
    'ENTERTAINMENT': 'Entertainment',
    'COMEDY': 'Entertainment',
    'PARENTING': 'Entertainment',
    'SPORTS': 'Entertainment',
    'CULTURE & ARTS': 'Entertainment',
    'MEDIA': 'Entertainment',
    'WEIRD NEWS': 'Entertainment',
    'STYLE': 'Entertainment',
    'STYLE & BEAUTY': 'Entertainment',
    'TASTE': 'Entertainment',
    'TRAVEL': 'Entertainment',
    'WEDDINGS': 'Entertainment',
    'ARTS': 'Entertainment',
    'ARTS & CULTURE': 'Entertainment',
    'FOOD & DRINK': 'Entertainment',
    'GOOD NEWS': 'Entertainment',
    'GREEN': 'Entertainment',
    'HOME & LIVING': 'Entertainment',
    
    'EDUCATION': 'Education',
    'COLLEGE': 'Education',
    'PARENTS': 'Education',
    
    'TECH': 'Tech',
    'SCIENCE': 'Tech',
    'BUSINESS': 'Tech',
    
    'CRIME': 'Crime',
    'BLACK VOICES': 'Crime',
    'POLITICS': 'Crime',
    'U.S. NEWS': 'Crime',
    'WORLD NEWS': 'Crime',
    'WORLDPOST': 'Crime',
    'ENVIRONMENT': 'Crime',
    'WELLNESS': 'Crime',
    'HEALTHY LIVING': 'Crime',
    'QUEER VOICES': 'Crime',
    'WOMEN': 'Crime',
    'LATINO VOICES': 'Crime',
    'RELIGION': 'Crime',
    'IMPACT': 'Crime',
    'DIVORCE': 'Crime',
    'FIFTY': 'Crime',
    'MONEY': 'Crime'
}

news_df_clean['category'] = news_df_clean['category'].map(
    lambda x: category_mapping.get(x, 'Entertainment')
)

# Keep only the 4 categories from assignment
valid_categories = ['Entertainment', 'Education', 'Tech', 'Crime']
news_df_clean = news_df_clean[news_df_clean['category'].isin(valid_categories)]

print(f"\nCleaned News Articles: {news_df_clean.shape}")
print("\nNews Category Distribution:")
print(news_df_clean['category'].value_counts())


Cleaned News Articles: (209521, 6)

News Category Distribution:
category
Crime            100774
Entertainment     92333
Tech              10301
Education          6113
Name: count, dtype: int64


In [9]:
# Filling missing age with median
train_users_clean = train_users.copy()
test_users_clean = test_users.copy()

age_median = train_users_clean['age'].median()
train_users_clean['age'].fillna(age_median, inplace=True)
test_users_clean['age'].fillna(age_median, inplace=True)

/var/folders/jx/8n7khtqd3kj995b8dsj6j8bw0000gn/T/ipykernel_42896/1834028968.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_users_clean['age'].fillna(age_median, inplace=True)
/var/folders/jx/8n7khtqd3kj995b8dsj6j8bw0000gn/T/ipykernel_42896/1834028968.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting va

In [10]:
# Encoding categorical features
# Region code encoding - FIT ON BOTH TRAIN AND TEST to avoid unseen labels
all_region_codes = pd.concat([train_users_clean['region_code'], 
                               test_users_clean['region_code']]).unique()
le_region = LabelEncoder()
le_region.fit(all_region_codes)

train_users_clean['region_code_encoded'] = le_region.transform(train_users_clean['region_code'])
test_users_clean['region_code_encoded'] = le_region.transform(test_users_clean['region_code'])

# Subscriber encoding (Boolean to int)
train_users_clean['subscriber_encoded'] = train_users_clean['subscriber'].astype(int)
test_users_clean['subscriber_encoded'] = test_users_clean['subscriber'].astype(int)

# Encode user labels for classification
le_user = LabelEncoder()
train_users_clean['label_encoded'] = le_user.fit_transform(train_users_clean['label'])

print(f"\nCleaned Train Users: {train_users_clean.shape}")
print(f"Cleaned Test Users: {test_users_clean.shape}")
print("\nUser Label Mapping:")
for i, label in enumerate(le_user.classes_):
    print(f"{i}: {label}")


Cleaned Train Users: (2000, 36)
Cleaned Test Users: (2000, 34)

User Label Mapping:
0: user_1
1: user_2
2: user_3


## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [19]:
## Selecting features and target for classification
exclude_cols = ['user_id', 'region_code', 'subscriber', 'label', 'browser_version']
feature_cols = [col for col in train_users_clean.columns 
                if col not in exclude_cols and col not in ['label_encoded', 'region_code_encoded', 'subscriber_encoded']]

feature_cols.extend(['region_code_encoded', 'subscriber_encoded'])

X = train_users_clean[feature_cols]
y = train_users_clean['label_encoded']   

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(f"\nTraining set size: {X_train.shape}")
print(f"Validation set size: {X_val.shape}")


Training set size: (1600, 30)
Validation set size: (400, 30)


In [17]:
classifiers = {
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=5000, solver='lbfgs'),  
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100)
}


results = {}

print("=" * 50)
print("CLASSIFIER TRAINING AND EVALUATION")
print("=" * 50)

for name, clf in classifiers.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_val_scaled)
    accuracy = accuracy_score(y_val, y_pred)
    results[name] = {'model': clf, 'accuracy': accuracy, 'predictions': y_pred}
    print(f"\n{name}: Accuracy = {accuracy:.4f}")

# Select best classifier
best_classifier_name = max(results, key=lambda x: results[x]['accuracy'])
best_classifier = results[best_classifier_name]['model']
best_accuracy = results[best_classifier_name]['accuracy']

print(f"\n{'=' * 50}")
print(f"BEST CLASSIFIER: {best_classifier_name} (Accuracy: {best_accuracy:.4f})")
print(f"{'=' * 50}")

CLASSIFIER TRAINING AND EVALUATION

Decision Tree: Accuracy = 0.8600

Logistic Regression: Accuracy = 0.8175

Random Forest: Accuracy = 0.8975

BEST CLASSIFIER: Random Forest (Accuracy: 0.8975)


In [22]:
print(f"Classification Report for Best Classifier: {best_classifier_name}\n")
print(classification_report(y_val, results[best_classifier_name]['predictions'], target_names=le_user.classes_))

Classification Report for Best Classifier: Random Forest

              precision    recall  f1-score   support

      user_1       0.89      0.86      0.87       142
      user_2       0.98      0.87      0.92       142
      user_3       0.83      0.97      0.90       116

    accuracy                           0.90       400
   macro avg       0.90      0.90      0.90       400
weighted avg       0.90      0.90      0.90       400



# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
